## Assignment 2 

### Machine Learning

#### Bits ID - 2026AJ05376

In [12]:
pip list | grep -i -E "streamlit|scikit|pandas"

pandas                          2.2.3
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install scikit-learn pandas numpy matplotlib seaborn joblib streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 101.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 kB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 MB 151.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 128.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 146.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 117.6 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.11.0
    Uninstalling typing_extensions-4.11.0:
      Successfully uninstalled typing_extensions-4.11.0
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.3
    Uninstalling protobuf-4.25.3:
      Successfully uninstalled protobuf-4.25.3
  Attempting uninstall: numpy
    Found existing installation: nu

In [1]:
import sys, sklearn, pandas, numpy
print(sys.executable)
print("sklearn", sklearn.__version__)
print("pandas", pandas.__version__)
print("numpy", numpy.__version__)

/home/cloud/anaconda3/bin/python
sklearn 1.9.1
pandas 2.2.3
numpy 2.5.4


In [5]:
import os
import pandas as pd
from sklearn.datasets import load_breast_cancer

# Make two folders in the current directory:
# "data" will hold the raw CSV, "model" will hold the saved models later.
# exist_ok=True means no error if the folders already exist.
os.makedirs("data", exist_ok=True)
os.makedirs("model", exist_ok=True)

# Load the Wisconsin breast cancer data that ships with scikit-learn.
# as_frame=True gives a pandas DataFrame, which is easier to work with than raw arrays.
d = load_breast_cancer(as_frame=True)

# .copy() so that changes below don't touch the original loaded object
df = d.frame.copy()

# sklearn labels it the opposite way round: 0 = malignant, 1 = benign.
# I want 1 = malignant, because in cancer screening the "positive" case is the
# cancer. Precision and recall are then measured on malignant cases, which is the
# meaningful one. 1 - target flips 0 to 1 and 1 to 0.
df["target"] = 1 - df["target"]

# Save to CSV so the project is reproducible and the dataset is in the repo.
# index=False stops pandas writing an extra row-number column.
df.to_csv("data/data.csv", index=False)

# --- Sanity checks: print things so I can confirm the data is what I expect ---

# Where the files were saved. This matters later for Git and the app.
print("Working directory:", os.getcwd())

# (rows, columns). Should be 569 rows, 31 columns (30 features + target)
print("Shape:", df.shape)

# How many benign (0) and malignant (1) cases. Shows the class balance.
print("Class counts (1 = malignant):", df["target"].value_counts().to_dict())

# Total count of empty cells. int() just makes the output cleaner.
print("Missing values:", int(df.isna().sum().sum()))

# The last 3 column names. The last one should be "target".
print("Last 3 columns:", list(df.columns[-3:]))

Working directory: /home/s2026aj05376/workspace
Shape: (569, 31)
Class counts (1 = malignant): {0: 357, 1: 212}
Missing values: 0
Last 3 columns: ['worst symmetry', 'worst fractal dimension', 'target']


In [7]:
%%writefile evaluation.py
# evaluation.py
# One place for the metric calculations so training and the app never disagree.

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
)


def score_model(model, X, y):
    """Return the 6 assignment metrics for a fitted model on data (X, y)."""

    # Hard class predictions (0 or 1). Used for the metrics that need a decision.
    pred = model.predict(X)

    # Probability of class 1 (malignant). AUC needs a score, not a hard 0/1,
    # because it measures how well the model ranks malignant above benign.
    # [:, 1] picks the "class 1" column of the probability table.
    prob = model.predict_proba(X)[:, 1]

    return {
        "Accuracy": accuracy_score(y, pred),     # share of all predictions that were right
        "AUC": roc_auc_score(y, prob),           # ranking quality, 0.5 = guessing, 1.0 = perfect
        "Precision": precision_score(y, pred),   # of those flagged malignant, how many truly were
        "Recall": recall_score(y, pred),         # of the truly malignant, how many we caught
        "F1": f1_score(y, pred),                 # balance between precision and recall
        "MCC": matthews_corrcoef(y, pred),       # one balanced score that uses all 4 confusion-matrix cells
    }


Writing evaluation.py


In [9]:
import os
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from evaluation import score_model

# 1. Confirm the file really was saved in the working folder
print("evaluation.py exists:", os.path.exists("evaluation.py"))

# 2. Load our CSV and split into features (X) and label (y)
df = pd.read_csv("data/data.csv")
X, y = df.drop(columns="target"), df["target"]

# 3. A quick split and a quick model, just to test the function
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=1)
quick = LogisticRegression(max_iter=5000).fit(X_tr, y_tr)

# 4. Run our metric function and print each value
for name, value in score_model(quick, X_te, y_te).items():
    print(f"{name}: {value:.4f}")


evaluation.py exists: True
Accuracy: 0.9474
AUC: 0.9940
Precision: 0.9737
Recall: 0.8810
F1: 0.9250
MCC: 0.8872


In [11]:
%%writefile train_models.py
# train_models.py
# Trains the 5 classifiers on the breast cancer data and saves everything the app needs.

import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from evaluation import score_model

# Random seed: controls the train/test split and the models that use randomness.
# CHANGE THIS to your own number so your results are unique to you.
SEED = 2026

# ---- 1. Load the data we saved in Step 2 ----
df = pd.read_csv("data/data.csv")
X = df.drop(columns="target")   # the 30 measurements
y = df["target"]                # 1 = malignant, 0 = benign
print("Data:", df.shape, "| classes:", y.value_counts().to_dict())

# ---- 2. Split into train (80%) and test (20%) ----
# stratify=y keeps the benign/malignant ratio the same in both parts,
# otherwise a small test set could end up with too few malignant cases.
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
print("Train rows:", len(X_tr), "| Test rows:", len(X_te))

# ---- 3. Save the test set as test_data.csv ----
# This is the file we upload into the Streamlit app later.
# It has the features AND the target column, as the assignment requires.
test_out = X_te.copy()
test_out["target"] = y_te
test_out.to_csv("test_data.csv", index=False)

# ---- 4. Define the 5 models ----
# (display name, file name, the classifier itself)
# Feel free to tweak these settings. Different choices give different results.
model_list = [
    ("Logistic Regression", "logistic_regression", LogisticRegression(max_iter=2000)),
    ("Decision Tree",       "decision_tree",       DecisionTreeClassifier(max_depth=5, random_state=SEED)),
    ("KNN",                 "knn",                 KNeighborsClassifier(n_neighbors=7)),
    ("Naive Bayes",         "naive_bayes",         GaussianNB()),
    # probability=True is needed so SVC can output probabilities for the AUC score
    ("Support Vector Machine", "svm",              SVC(kernel="rbf", C=1.0, probability=True, random_state=SEED)),
]

# ---- 5. Train, evaluate and save each model ----
results = []
for display_name, file_name, clf in model_list:

    # A Pipeline = scaler + classifier packed together.
    # Scaling matters for KNN, SVM and Logistic Regression because the features
    # are on very different ranges (area is in the hundreds, smoothness is below 1).
    # Saving the whole pipeline means the app doesn't have to scale anything itself.
    pipe = Pipeline([("scaler", StandardScaler()), ("clf", clf)])

    # Learn from the TRAINING data only
    pipe.fit(X_tr, y_tr)

    # Save the trained pipeline so the app can load it without retraining
    joblib.dump(pipe, f"model/{file_name}.joblib")

    # Score on the TEST data, which the model has never seen
    scores = score_model(pipe, X_te, y_te)
    results.append({"ML Model Name": display_name, **scores})
    print("Trained and saved:", display_name)

# ---- 6. Build the comparison table and save it ----
results_df = pd.DataFrame(results).round(4)
results_df.to_csv("model/metrics.csv", index=False)

print("\n=== Comparison table (test set) ===")
print(results_df.to_string(index=False))


Writing train_models.py


In [13]:
%run train_models.py

Data: (569, 31) | classes: {0: 357, 1: 212}
Train rows: 455 | Test rows: 114
Trained and saved: Logistic Regression
Trained and saved: Decision Tree
Trained and saved: KNN
Trained and saved: Naive Bayes
Trained and saved: Support Vector Machine

=== Comparison table (test set) ===
         ML Model Name  Accuracy    AUC  Precision  Recall     F1    MCC
   Logistic Regression    0.9737 0.9987     0.9756  0.9524 0.9639 0.9433
         Decision Tree    0.9561 0.9590     0.9111  0.9762 0.9425 0.9085
                   KNN    0.9737 0.9974     0.9535  0.9762 0.9647 0.9439
           Naive Bayes    0.9561 0.9964     0.9302  0.9524 0.9412 0.9064
Support Vector Machine    0.9825 1.0000     0.9545  1.0000 0.9767 0.9633


/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [15]:
import os, joblib
import pandas as pd
from evaluation import score_model

# 1. Did all the files get created?
print("Files in model/:", sorted(os.listdir("model")))

# 2. Is test_data.csv the right shape? (expect 114 rows, 31 columns incl. target)
t = pd.read_csv("test_data.csv")
print("test_data.csv shape:", t.shape, "| last column:", t.columns[-1])

# 3. Reload ONE saved model from disk and score it on test_data.csv.
#    This should match the SVM row in the table above, which proves that
#    loading from file (what the app does) gives the same numbers.
m = joblib.load("model/svm.joblib")
print({k: round(v, 4) for k, v in score_model(m, t.drop(columns="target"), t["target"]).items()})


Files in model/: ['decision_tree.joblib', 'knn.joblib', 'logistic_regression.joblib', 'metrics.csv', 'naive_bayes.joblib', 'svm.joblib']
test_data.csv shape: (114, 31) | last column: target
{'Accuracy': 0.9825, 'AUC': 1.0, 'Precision': 0.9545, 'Recall': 1.0, 'F1': 0.9767, 'MCC': 0.9633}


In [17]:
import pandas as pd

# Read the saved comparison table back from disk and print it
table = pd.read_csv("model/metrics.csv")
print(table.to_string(index=False))

# Check that nothing is empty (expect 0)
print("Missing cells:", int(table.isna().sum().sum()))


         ML Model Name  Accuracy    AUC  Precision  Recall     F1    MCC
   Logistic Regression    0.9737 0.9987     0.9756  0.9524 0.9639 0.9433
         Decision Tree    0.9561 0.9590     0.9111  0.9762 0.9425 0.9085
                   KNN    0.9737 0.9974     0.9535  0.9762 0.9647 0.9439
           Naive Bayes    0.9561 0.9964     0.9302  0.9524 0.9412 0.9064
Support Vector Machine    0.9825 1.0000     0.9545  1.0000 0.9767 0.9633
Missing cells: 0


In [19]:
# Check that the extra libraries the app needs are installed
import importlib

for pkg in ["streamlit", "matplotlib", "seaborn", "joblib"]:
    try:
        mod = importlib.import_module(pkg)
        print(pkg, "OK, version", mod.__version__)
    except ModuleNotFoundError:
        print(pkg, "MISSING -> run: %pip install", pkg)


streamlit OK, version 1.67.0
matplotlib OK, version 3.10.1
seaborn OK, version 0.13.2
joblib OK, version 1.4.2


In [21]:
%%writefile app.py
# app.py
# Streamlit front-end: upload test data, pick a model, see metrics and a confusion matrix.
# The models are LOADED from disk. Nothing is retrained here.

import joblib
import pandas as pd
import streamlit as st
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

from evaluation import score_model   # same metric function used during training

# Name of the label column in the CSV the user uploads
TARGET = "target"

# Dropdown label -> saved model file.
# These paths are relative, so run the app from the project folder.
MODEL_FILES = {
    "Logistic Regression": "model/logistic_regression.joblib",
    "Decision Tree": "model/decision_tree.joblib",
    "KNN": "model/knn.joblib",
    "Naive Bayes": "model/naive_bayes.joblib",
    "Support Vector Machine": "model/svm.joblib",
}

# ---- Page setup ----  (change the title and icon to make it yours)
st.set_page_config(page_title="Tumour Classifier Comparison", page_icon="🧬", layout="wide")
st.title("🧬 Tumour Classifier Comparison")
st.write(
    "Upload the **test CSV** (features plus a `target` column where 1 = malignant). "
    "The five pre-trained models are loaded from disk and scored on your file. "
    "No retraining happens in this app."
)


# cache_resource keeps a loaded model in memory, so switching models or
# interacting with the page doesn't reload the file from disk every time.
@st.cache_resource
def load_model(path):
    return joblib.load(path)


# ---- Sidebar controls ----
st.sidebar.header("Controls")
uploaded = st.sidebar.file_uploader("1) Upload test data (CSV)", type="csv")
choice = st.sidebar.selectbox("2) Choose a model", list(MODEL_FILES.keys()))

# Nothing to show until a file is uploaded, so stop here with a hint.
if uploaded is None:
    st.info("👈 Upload `test_data.csv` from the repo in the sidebar to begin.")
    st.stop()

# ---- Read and validate the uploaded file ----
data = pd.read_csv(uploaded)
model = load_model(MODEL_FILES[choice])

# The saved pipeline remembers which feature columns it was trained on.
needed_cols = list(model.feature_names_in_)
missing = [c for c in needed_cols + [TARGET] if c not in data.columns]
if missing:
    st.error(f"Your file is missing {len(missing)} required column(s), e.g. {missing[:3]}")
    st.stop()

X = data[needed_cols]   # only the feature columns, in the order the model expects
y = data[TARGET]        # the true labels

# AUC and MCC can't be computed if the file contains only one class.
if y.nunique() < 2:
    st.error("The target column has only one class, so metrics can't be computed. Upload the full test set.")
    st.stop()

with st.expander("Preview uploaded data"):
    st.write(f"{len(data)} rows, {len(needed_cols)} features")
    st.dataframe(data.head())

# ---- Two tabs: selected model, and all models side by side ----
tab_one, tab_all = st.tabs(["Selected model", "Compare all models"])

with tab_one:
    st.subheader(f"Results for: {choice}")

    # Six metric cards in a row
    scores = score_model(model, X, y)
    cols = st.columns(len(scores))
    for col, (name, value) in zip(cols, scores.items()):
        col.metric(name, f"{value:.4f}")

    left, right = st.columns(2)
    preds = model.predict(X)

    with left:
        st.markdown("**Confusion matrix**")
        # labels=[0, 1] fixes the row/column order, so it always reads Benign then Malignant
        cm = confusion_matrix(y, preds, labels=[0, 1])
        fig, ax = plt.subplots(figsize=(4, 3))
        sns.heatmap(
            cm, annot=True, fmt="d", cmap="Purples", cbar=False, ax=ax,
            xticklabels=["Benign", "Malignant"], yticklabels=["Benign", "Malignant"],
        )
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
        st.pyplot(fig)

    with right:
        st.markdown("**Classification report**")
        st.code(classification_report(y, preds, target_names=["Benign", "Malignant"]))

with tab_all:
    st.subheader("All five models on the uploaded file")
    rows = []
    for name, path in MODEL_FILES.items():
        m = load_model(path)
        rows.append({"Model": name, **score_model(m, data[list(m.feature_names_in_)], y)})
    st.dataframe(pd.DataFrame(rows).set_index("Model").round(4))


Writing app.py


In [23]:
import py_compile
py_compile.compile("app.py", doraise=True)
print("app.py compiles OK")


app.py compiles OK


In [25]:
import pandas as pd

# Copy of the test file with the label column removed, so the app should refuse it
pd.read_csv("test_data.csv").drop(columns="target").to_csv("bad_test.csv", index=False)
print("bad_test.csv created")


bad_test.csv created


In [27]:
import os
os.remove("bad_test.csv")
print("bad_test.csv removed:", not os.path.exists("bad_test.csv"))


bad_test.csv removed: True


In [29]:
import sys
from importlib.metadata import version   # reads the installed version of a package

# Every third-party library that app.py, evaluation.py and train_models.py import.
# (Names here are the pip names, e.g. "scikit-learn" and not "sklearn".)
packages = ["streamlit", "scikit-learn", "numpy", "pandas",
            "matplotlib", "seaborn", "joblib"]

# Build lines like "pandas==2.2.3" using whatever is installed right now
lines = [f"{p}=={version(p)}" for p in packages]

# Write them to requirements.txt, one per line
with open("requirements.txt", "w") as f:
    f.write("\n".join(lines) + "\n")

# Show what was written, plus the Python version.
# We'll need the Python version when deploying.
print("Python:", sys.version.split()[0])
print("---- requirements.txt ----")
print(open("requirements.txt").read())


Python: 3.12.7
---- requirements.txt ----
streamlit==1.67.0
scikit-learn==1.9.1
numpy==2.5.4
pandas==2.2.3
matplotlib==3.10.1
seaborn==0.13.2
joblib==1.4.2



In [1]:
%%writefile README.md
# Breast Cancer Classification: Five-Model Comparison

**Live Streamlit app:** https://breastcancermodelcomparison-bitspilani-mudhukrishna.streamlit.app/

**GitHub repository:** https://github.com/Krishna14547/BreastCancerModelComparison

## a. Problem statement

Predict whether a breast mass is **malignant or benign** from 30 numeric measurements of cell nuclei. This is a binary classification problem, and the positive class (1) is malignant. Five classifiers are trained on the same data and compared on the same held-out test set using Accuracy, AUC, Precision, Recall, F1 and MCC. The trained models are served through an interactive Streamlit app.

## b. Dataset description

- **Source:** Breast Cancer Wisconsin (Diagnostic) dataset from the UCI Machine Learning Repository, loaded through the copy bundled with scikit-learn (`load_breast_cancer`).
- **Instances:** 569 (357 benign, 212 malignant).
- **Features:** 30 numeric features. They are the mean, standard error and worst value of 10 measurements taken from digitised images of fine-needle aspirates of breast masses: radius, texture, perimeter, area, smoothness, compactness, concavity, concave points, symmetry and fractal dimension.
- **Target:** `target`, where 1 = malignant and 0 = benign. scikit-learn encodes this the other way round, so the labels were flipped to make malignant the positive class.
- **Data quality:** no missing values.
- **Preprocessing:** features are standardised with `StandardScaler` inside each model pipeline, so scaling is learned from training data only.
- **Split:** 80% train (455 rows) and 20% test (114 rows), stratified on the target, random seed 2026.

## c. GitHub repository

https://github.com/Krishna14547/BreastCancerModelComparison

- `app.py`: Streamlit app
- `train_models.py`: trains the five models and saves them
- `evaluation.py`: metric calculations shared by training and the app
- `requirements.txt`: pinned dependencies
- `test_data.csv`: held-out test data (features plus `target`)
- `data/data.csv`: full dataset
- `model/`: saved pipelines (`.joblib`) for all five classifiers and `metrics.csv`

To run locally: install the packages in `requirements.txt`, then run `streamlit run app.py` from the project folder.

## d. Models used

Each model is a pipeline of `StandardScaler` followed by the classifier. Settings: Logistic Regression (max_iter=2000), Decision Tree (max_depth=5), KNN (k=7), Gaussian Naive Bayes (defaults), SVM (RBF kernel, C=1.0, probability enabled).

### Comparison table (test set, 114 rows)

| ML Model Name | Accuracy | AUC | Precision | Recall | F1 | MCC |
|---|---|---|---|---|---|---|
| Logistic Regression | 0.9737 | 0.9987 | 0.9756 | 0.9524 | 0.9639 | 0.9433 |
| Decision Tree | 0.9561 | 0.9590 | 0.9111 | 0.9762 | 0.9425 | 0.9085 |
| KNN | 0.9737 | 0.9974 | 0.9535 | 0.9762 | 0.9647 | 0.9439 |
| Naive Bayes | 0.9561 | 0.9964 | 0.9302 | 0.9524 | 0.9412 | 0.9064 |
| Support Vector Machine | 0.9825 | 1.0000 | 0.9545 | 1.0000 | 0.9767 | 0.9633 |

### Observations on model performance

| ML Model Name | Observation about model performance |
|---|---|
| Logistic Regression | It reaches 0.9737 accuracy and an AUC of 0.9987. It has the highest precision (0.9756) but a lower recall (0.9524), so it misses a few malignant cases while rarely raising false alarms. |
| Decision Tree | It is the weakest on AUC (0.9590) and accuracy (0.9561). Its precision is the lowest (0.9111), which means more benign cases are flagged as malignant, even though recall is high (0.9762). |
| KNN | It matches Logistic Regression on accuracy (0.9737) and has an MCC of 0.9439, the second best. Its recall (0.9762) is higher than Logistic Regression's, with an AUC of 0.9974. |
| Naive Bayes | It shares the lowest accuracy (0.9561) and has the lowest MCC (0.9064), but its AUC is high (0.9964). Precision (0.9302) and recall (0.9524) are balanced, so it is a reasonable but not leading option. |
| Support Vector Machine | It scores best on every headline metric: accuracy 0.9825, AUC 1.0000, F1 0.9767 and MCC 0.9633. It also has perfect recall (1.0000), so it misses no malignant cases. |
| **Overall, winner for this dataset?** | Support Vector Machine (RBF). It has the highest accuracy, AUC, F1 and MCC. Its recall of 1.0000 matters most here, because a missed malignant case is the costliest error. Its precision (0.9545) is only slightly below Logistic Regression's. The test set has only 114 rows, so the gaps between the top models are small and may not hold on other splits. |


Writing README.md


In [3]:
# Read the README back and run some simple checks
text = open("README.md").read()

# 1. Any placeholder left? (must be 0)
print("Placeholders left:", text.count("<<"))

# 2. Are the key sections and all five model names present?
for needle in ["a. Problem statement", "b. Dataset description", "c. GitHub repository",
               "d. Models used", "Overall, winner",
               "Logistic Regression", "Decision Tree", "KNN", "Naive Bayes", "Support Vector Machine"]:
    print(f"{needle!r:35} found: {needle in text}")

# 3. Do both links appear?
print("App link present:", "streamlit.app" in text)
print("Repo link present:", "github.com/Krishna14547/BreastCancerModelComparison" in text)


Placeholders left: 0
'a. Problem statement'              found: True
'b. Dataset description'            found: True
'c. GitHub repository'              found: True
'd. Models used'                    found: True
'Overall, winner'                   found: True
'Logistic Regression'               found: True
'Decision Tree'                     found: True
'KNN'                               found: True
'Naive Bayes'                       found: True
'Support Vector Machine'            found: True
App link present: True
Repo link present: True
